In [1]:
pip install rugbypy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
from rugbypy.match import *
from rugbypy.competition import *
from rugbypy.team import *
from rugbypy.venue import *
from rugbypy.player import *

In [4]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [5]:
## Used to eliminate dumb warnings

pd.options.mode.chained_assignment = None

In [6]:
## Reading in data

test_matches = pd.read_excel("detailed_test_matches.xlsx") ## Description of match contexts and the final score of each match
match_stats = pd.read_excel("aggregated_stats.xlsx") ## Match stats collected using player data instead of team data
test_nations = pd.read_excel("test_nations.xlsx") ## List of all Tier 1 Test Nations
team_stats = pd.read_excel("team_stats.xlsx") ## Match stats by team
venues = fetch_all_venues() ## All venues in the database

Fetching all venues...


In [7]:
## Only including test matches between Tier 1 Nations

test_matches = test_matches[test_matches['home_team'].isin(test_nations['team_name']) &
                            test_matches['away_team'].isin(test_nations['team_name'])]

In [8]:
## Removal of Fiji - no national stadium where they're the home team

test_matches = test_matches[test_matches['home_team'] != "Fiji"]

In [9]:
## Data Cleaning

test_match_results = test_matches[['match_id', 'date', 'season', 'competition_id', 'competition', 'venue_id', 'venue', 
                                    'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_score', 'away_score']]

test_match_results['winner'] = np.where(test_match_results['home_score'] == test_match_results['away_score'],
                                "Draw", np.where(test_match_results['home_score'] > test_match_results['away_score'], 
                                test_match_results['home_team'], test_match_results['away_team']))

test_match_results['home_team_win'] = np.where(test_match_results['winner'] == test_match_results['home_team'], 1, 0)

test_match_results['neutral_match'] = np.where((test_match_results['competition'] == "Rugby World Cup") &
                                                (test_match_results['home_team'] != "France"),
                                                "neutral", "NA")

## test_match_results - information on all test matches including tests where there is no clear home team (neutral match)

home_away_tests = test_match_results[test_match_results['neutral_match'] != "neutral"]

## home_away_tests

## home_away_tests - test matches where there is a clear home/away distinction. Ignores neutral-site 


In [10]:
## Removal of New Zealand v. South Africa at Twickenham (neutral-site RWC Preparation Match)

home_away_tests = home_away_tests[~home_away_tests['match_id'].isin(['db7135be'])]

In [11]:
## Only included venues that have hosted a test match

test_venues = venues[venues['venue_id'].isin(test_match_results['venue_id'])]

## Validation of the venues list; removed Wembley Stadium and Soldier Field

test_venues = test_venues[~test_venues['venue_name'].isin(['Wembley Stadium', 'Soldier Field'])] 

In [12]:
## Only included match stats for Test 1 Teams

match_stats = match_stats[match_stats['team'].isin(test_nations['team_name'])]

In [13]:
## Data validation

match_stats = match_stats.rename(columns = {"team": "team_name"})

match_stats = match_stats[match_stats['turnovers_lost'] < 30] ## Some matches had teams who committed 100+ turnovers 


In [14]:
## Merging test nations with their associated match stats

match_stats = pd.merge(test_nations, match_stats, how = 'inner', on = 'team_name')

## match_stats


In [15]:
## Joining data columns of stats not collected in the dataset using player data

team_stats = team_stats[['team_id', 'team', 'game_date', 'team_vs', 'team_vs_id', 'match_id', '22m_entries', '22m_conversion',
                        'post_contact_metres', 'ruck_speed_0_3_pct', 'ruck_speed_3_6_pct', 'ruck_speed_6_plus_pct',
                        'possession', 'territory', 'scrums_total', 'mauls_total', 'rucks_total']]

team_stats = team_stats.rename(columns={"team": "team_name"}) # renamed for the purpose of joining

In [16]:
## Join this with data collected from players data set

potential_model_set = pd.merge(team_stats, match_stats, how="left", on = ['team_id', 'team_name', 'match_id'])

In [17]:
## Home/Away Designation

## Intended columns to drop; all associated with each team and associated score. game_date is a redundant column. 

drop_cols = ['team_name', 'team_vs', 'team_vs_id', 'game_date', 'score', 'team_vs_score']

potential_model_set.drop(columns = [c for c in drop_cols if c in potential_model_set.columns])

stat_cols = [c for c in potential_model_set.columns if c not in ('team_id', 'match_id')]

## Creates a new data frame that adds the home designation for the statistics associated with the home team

home_team_stats = potential_model_set.rename(columns={
    'team_id': 'home_team_id',
    **{c: f'home_{c}' for c in stat_cols}
})

## Creates a new data frame that adds the away designation for the statistics associated with the away team

away_team_stats = potential_model_set.rename(columns={
    'team_id': 'away_team_id',
    **{c: f'away_{c}' for c in stat_cols}
})

## Joins both data sets

potential_model_set = pd.merge(test_match_results, home_team_stats, on = ['match_id', 'home_team_id'], how = 'left')

potential_model_set = potential_model_set.merge(away_team_stats, on = ['match_id', 'away_team_id'], how = 'left')

In [18]:
## Removing Neutral Matches

potential_model_set = potential_model_set[potential_model_set['neutral_match'] != "neutral"]

potential_model_set = potential_model_set.drop(columns=['neutral_match', 'home_game_date', 'away_game_date'])



In [19]:
## Eventually will delete later
match_info = potential_model_set[['match_id', 'date', 'season', 'competition_id', 'competition', 
                                'venue_id', 'venue', 'home_team_id', 'home_team', 'away_team_id'
                                ,'away_team', 'home_score', 'away_score', 'winner', 'home_team_win']]


game_stats = potential_model_set.drop(columns = ['season', 'competition_id', 'competition', 'venue_id', 'venue',
                                    'home_score', 'away_score', 'winner', 'home_team_win', 'home_team_vs_id', 'away_team_vs_id',
                                    'home_team_name', 'home_team_vs', 'away_team_name', 'away_team_vs'])



In [20]:
## Adding potential features to model

model_set = potential_model_set[['match_id', 'date', 'season', 'competition_id', 'competition','venue_id', 'venue', 
                                        'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_score', 'away_score',
                                        'winner', 'home_team_win', 'home_carries', 'away_carries', 'home_line_breaks',
                                        'away_line_breaks', 'home_tackles_made', 'away_tackles_made', 'home_tackles_missed',
                                        'away_tackles_missed', 'home_kicks', 'away_kicks', 'home_metres_carried',
                                        'away_metres_carried', 'home_defenders_beaten', 'away_defenders_beaten', 
                                        'home_lineouts_won', 'away_lineouts_won', 'home_total_lineouts', 'away_total_lineouts',
                                        'home_passes', 'away_passes', 'home_yellow_cards', 'away_yellow_cards', 
                                        'home_red_cards', 'away_red_cards', 'home_tries', 'away_tries', 'home_defenders_beaten',
                                        'away_defenders_beaten', 'home_turnovers_won', 'away_turnovers_won', 'home_turnovers_conceded',
                                        'away_turnovers_conceded', 'home_penalties_conceded', 'away_penalties_conceded', 'home_meters_run',
                                        'away_meters_run', 'home_penalty_goals', 'away_penalty_goals']]

In [21]:
model_set = model_set.dropna()

model_set

,match_id,date,season,competition_id,competition,venue_id,venue,home_team_id,home_team,away_team_id,...,home_turnovers_won,away_turnovers_won,home_turnovers_conceded,away_turnovers_conceded,home_penalties_conceded,away_penalties_conceded,home_meters_run,away_meters_run,home_penalty_goals,away_penalty_goals
0,26eb0d57,20220205,2022,7a1361e7,Six Nations,07b0586c,Aviva Stadium,165b36e5,Ireland,094dae60,...,0.0,0.0,19.0,16.0,5.0,13.0,519.0,218.0,1.0,0.0
1,d803469c,20220205,2022,7a1361e7,Six Nations,a35811c5,Scottish Gas Murrayfield,4409215c,Scotland,f60f757d,...,0.0,0.0,11.0,10.0,13.0,10.0,299.0,416.0,2.0,4.0
2,4a49c716,20220206,2022,7a1361e7,Six Nations,5b691aae,Stade de France,67d88578,France,d0cffe5e,...,0.0,0.0,10.0,17.0,14.0,10.0,492.0,265.0,2.0,1.0
3,7ec3e5e5,20220212,2022,7a1361e7,Six Nations,fe933f9f,Principality Stadium,094dae60,Wales,4409215c,...,0.0,0.0,7.0,11.0,8.0,13.0,462.0,466.0,4.0,4.0
4,71ccf2fb,20220212,2022,7a1361e7,Six Nations,5b691aae,Stade de France,67d88578,France,165b36e5,...,0.0,0.0,12.0,18.0,7.0,10.0,334.0,356.0,6.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,c09a6180,20260815,2026,96c775f2,International Test Match,eb9c53f6,Queensland Country Bank Stadium,b904fef9,Australia,f97333f3,...,8.0,4.0,0.0,0.0,5.0,13.0,0.0,0.0,0.0,0.0
262,c09a6180,20260815,2026,96c775f2,International Test Match,eb9c53f6,Queensland Country Bank Stadium,b904fef9,Australia,f97333f3,...,8.0,4.0,0.0,0.0,5.0,13.0,0.0,0.0,0.0,0.0
263,a215a140,20260822,2026,96c775f2,International Test Match,e952f544,Ellis Park,9b93d911,South Africa,d75e45cc,...,5.0,14.0,0.0,0.0,9.0,14.0,0.0,0.0,0.0,0.0
264,e1440199,20260829,2026,96c775f2,International Test Match,3ce3079c,DHL Stadium,9b93d911,South Africa,d75e45cc,...,6.0,2.0,0.0,0.0,5.0,8.0,0.0,0.0,0.0,0.0


In [47]:
## Elo Rating Development

def expected_score(rating_a, rating_b):
    return 1/(1 + 10 **((rating_b - rating_a)/400))

def update_elo(rating, expected, actual, k = 20):
    return rating + k * (actual - expected)

def run_elo(matches, win_pct_seeds, initial_rating = 1000, k = 20):
    matches = matches.sort_values('date').reset_index(drop = True)
    seed_lookup = win_pct_seeds.set_index(['team_id', 'seed_season'])['elo_seed'].to_dict()
    ratings = {}
    records = []
    current_season = None

    for x, row in matches.iterrows():
        home_id = row['home_team_id']
        away_id = row['away_team_id']
        season = row['season']

        if season != current_season:
            current_season = season
        for id in ratings:
            seed = seed_lookup.get((id, season), ratings[id])
            ratings[id] = seed
    
        ratings.setdefault(home_id, seed_lookup.get((home_id, season), initial_rating))
        ratings.setdefault(away_id, seed_lookup.get((away_id, season), initial_rating))

        r_home = ratings[home_id]
        r_away = ratings[away_id]

        ratings.setdefault(home_id, )

        p_home = expected_score(r_home, r_away)

        if row['home_score'] > row['away_score']:
            outcome = 1
        elif row['home_score'] == row['away_score']:
            outcome = 0.5
        else:
            outcome = 0
        
        records.append({
            'match_id': row['match_id'],
            'date': row['date'],
            'home_team_id': home_id,
            'away_team_id': away_id,
            'pre_elo_home': r_home,
            'pre_elo_away': r_away,
            'elo_diff': r_home - r_away,
            'predicted_prob': p_home,
            'actual_outcome': outcome,
            'wpa': outcome - p_home
        })

        ratings[home_id] = update_elo(r_home, p_home, outcome, k)
        ratings[away_id] = update_elo(r_away, 1 - p_home, 1 - outcome, k)

    results_df = pd.DataFrame(records)
    final_ratings = pd.Series(ratings, name = 'elo')

    return results_df, final_ratings

def elo_from_win_pct(win_pct, base=1000, scale=200):
    return base + scale * (win_pct - 0.5)



In [ ]:
## Adding season form to elo rating

test_matches['winner'] = np.where(test_matches['home_score'] == test_matches['away_score'],
                                "Draw", np.where(test_matches['home_score'] > test_matches['away_score'], 
                                test_matches['home_team'], test_matches['away_team']))

home_matches = test_matches[['season', 'home_team_id', 'home_team', 'home_score', 'away_score', 'winner']].copy()
home_matches['team_id'] = home_matches['home_team_id']
home_matches['team'] = home_matches['home_team']
home_matches['win'] = np.where(home_matches['home_team'] == home_matches['winner'], 1, 0)

away_matches = test_matches[['season', 'away_team_id', 'away_team', 'home_score', 'away_score', 'winner']].copy()
away_matches['team_id'] = away_matches['away_team_id']
away_matches['team'] = away_matches['away_team']
away_matches['win'] = np.where(away_matches['away_team'] == away_matches['winner'], 1, 0)

team_matches = pd.concat([
    home_matches[['season', 'team_id', 'team', 'win']],
    away_matches[['season', 'team_id', 'team', 'win']]
], ignore_index = True)

season_summary = (
    team_matches
    .groupby(['season', 'team_id', 'team'])
    .agg(
        games_played = ('team', 'size'),
        wins = ('win', 'sum')
    )
    .reset_index()
)

season_summary['win_pct'] = np.round((season_summary['wins']/season_summary['games_played']), 2)

season_summary['seed_season'] = season_summary['season'] + 1
season_summary['elo_seed'] = season_summary['win_pct'].apply(elo_from_win_pct)

In [ ]:
## Running the elo rating

results, final_ratings = run_elo(test_matches, season_summary, k = 20)

In [50]:
## Joining elo ratings with matches in the model set

model_set = model_set.merge(results, how = 'inner', on = ['match_id', 'date', 'home_team_id', 'away_team_id'])


